# 03 RAGAS-style Evaluation

Notebook này tập trung vào answer quality sau khi đã chọn retrieval method tốt nhất.

Evaluator hiện dùng AI Studio Gemini API qua `google-genai` để chấm theo phong cách RAGAS:
- `faithfulness`,
- `answer_relevancy`,
- `context_precision`.

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

ANSWERS_PATH = PROJECT_ROOT / "reports/rag_answers.json"
EVAL_PATH = PROJECT_ROOT / "reports/ragas_evaluation.json"

## Existing generated answers

File `reports/rag_answers.json` được tạo bằng method mặc định `hybrid_rrf` và model `gemini-3.1-flash-lite`.

In [ ]:
answers = json.loads(ANSWERS_PATH.read_text(encoding="utf-8"))
pd.DataFrame([
    {
        "question": row["question"],
        "answer_preview": row["answer"][:180],
        "num_contexts": len(row["contexts"]),
    }
    for row in answers
])

## RAGAS-style score table

In [ ]:
result = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
pd.DataFrame([result["aggregate"]])

In [ ]:
details = pd.DataFrame(result["details"])
details[["question", "faithfulness", "answer_relevancy", "context_precision", "reason"]]

## Re-run evaluation with AI Studio key

Chạy cell dưới đây nếu đã set `GEMINI_API_KEYS` hoặc `GEMINI_API_KEY` trong environment.

```bash
export GEMINI_API_KEYS=key_1,key_2
export GEMINI_MODEL=gemini-3.1-flash-lite
```

In [ ]:
if os.getenv("GEMINI_API_KEYS") or os.getenv("GEMINI_API_KEY"):
    from src.evaluation.evaluate_ragas import evaluate_with_ai_studio_genai
    fresh_result = evaluate_with_ai_studio_genai(answers, model=os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite"))
    pd.DataFrame([fresh_result["aggregate"]])
else:
    print("Set GEMINI_API_KEYS/GEMINI_API_KEY to re-run GenAI judging.")

## Selected model/method for future deploy

- Retrieval: **`hybrid_rrf`**
- Vector backend: **FAISS**
- LLM: **Gemini 3.1 Flash Lite** via AI Studio `google-genai`
- Key handling: **BatchGeminiClient** round-robin over `GEMINI_API_KEYS`

Thiết kế này giữ code đơn giản, dễ giải thích trong phỏng vấn, và vẫn đủ tốt để scale sang FastAPI/Streamlit sau này.